# Verify rung 0

Recomputes every reported number from files committed in this repository. No cluster access
needed. The cells use the standard library, pandas and matplotlib only — nothing from this
repository's own code — so what runs is what you read. Figures are drawn from the same tables
the checks use; no stored images.

Run all cells; about a minute.

```
uv sync --extra dev
uv run jupyter lab docs/tasks/rung0-replicate-ceiling/verify.ipynb
```

Two things cannot be checked here and are flagged where they arise: the gene and drug panel
files live on the cluster, pinned by checksum in the run record, and the 1,026 raw data files
live on cluster scratch, so their integrity reduces to the committed inventory's hash.

In [ ]:
import hashlib
import json
import re
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
repo = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
assert repo is not None, "run this notebook from inside the repository (its own folder works)"
task = repo / "docs" / "tasks" / "rung0-replicate-ceiling"
results = repo / "results" / "rung0-replicate-ceiling"
print("repository:", repo)

## 1. The result

Rung 0 asks how much of a drug's measured effect on gene expression is real rather than noise.
For each (cell line, drug) condition, the replicate plates are split into two groups, each
group's per-gene log2 fold change is averaged, and the two averages are correlated across the
gene panel. That is 1,600 conditions and 1,600 correlations. Every reported number summarises
them.

The correlation is a ceiling: a model predicting these responses cannot match them better than
they match themselves.

In [ ]:
per_pair = pd.read_csv(task / "rung0_per_pair_r.csv", keep_default_na=False)
headline = pd.read_csv(results / "rung0_delta_reproducibility.csv").iloc[0]
r = per_pair["r"].to_numpy(float)
print(per_pair.head().to_string(index=False))
print(f"\n{len(per_pair)} conditions, {np.isfinite(r).sum()} scored")

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.hist(r, bins=70, color="0.72", edgecolor="0.5", label="the 1,600 measured conditions")
ax.axvline(0, color="k", lw=1)
ax.axvspan(
    headline["splithalf_q1_r"],
    headline["splithalf_q3_r"],
    color="crimson",
    alpha=0.10,
    label=f"middle half ({headline['splithalf_q1_r']}-{headline['splithalf_q3_r']})",
)
ax.axvline(np.mean(r), color="crimson", lw=2.2, label=f"mean {np.mean(r):.3f} (the headline)")
ax.axvline(np.median(r), color="0.3", lw=1.5, label=f"median {np.median(r):.3f}")
ax.axvline(
    headline["null_same_drug_mean_r"],
    ls="--",
    color="darkorange",
    label=f"same-drug chance floor {headline['null_same_drug_mean_r']}",
)
ax.axvline(
    headline["null_diff_drug_mean_r"],
    ls="--",
    color="steelblue",
    label=f"mismatched chance floor {headline['null_diff_drug_mean_r']}",
)
ax.set_xlabel("agreement between two independent halves of the replicate plates (Pearson r)")
ax.set_ylabel("conditions")
ax.set_title("The rung-0 data: how well each drug response reproduces")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

Each reported statistic, recomputed from those 1,600 values and compared with the
published row.

In [ ]:
e = per_pair["mean_abs_delta"].to_numpy(float)
edges = np.quantile(e, [1 / 3, 2 / 3])
recomputed = {
    "splithalf_mean_r": round(float(np.mean(r)), 3),
    "splithalf_median_r": round(float(np.median(r)), 3),
    "splithalf_q1_r": round(float(np.quantile(r, 0.25)), 3),
    "splithalf_q3_r": round(float(np.quantile(r, 0.75)), 3),
    "frac_pos": round(float(np.mean(r > 0)), 3),
    "spearman_brown_full": round(2 * float(np.mean(r)) / (1 + float(np.mean(r))), 3),
    "n_pairs": len(per_pair),
}
bands = [(-np.inf, edges[0]), (edges[0], edges[1]), (edges[1], np.inf)]
for t, (lo, hi) in enumerate(bands, 1):
    recomputed[f"splithalf_mean_r_tercile{t}"] = round(float(np.mean(r[(e > lo) & (e <= hi)])), 3)

for key, value in recomputed.items():
    print(f"{key:26s} from the 1,600 values: {value:<8} reported: {headline[key]}")
    assert value == headline[key], f"{key} does not recompute"
print("\nevery reported statistic recomputes from the raw per-condition values")

The result file's SHA-256 was recorded when the job ran. Recomputing it from the file
just read shows the numbers have not been edited since.

In [ ]:
record = json.loads((results / "rung0_delta_reproducibility.provenance.json").read_text())

result_sha = hashlib.sha256((repo / record["result"]).read_bytes()).hexdigest()
print("fingerprint recorded at run time :", record["result_sha256"])
print("fingerprint of the file above    :", result_sha)
assert result_sha == record["result_sha256"]

log_sha = hashlib.sha256((repo / record["log"]).read_bytes()).hexdigest()
print("job-log fingerprint, recorded    :", record["log_sha256"])
print("job-log fingerprint, recomputed  :", log_sha)
assert log_sha == record["log_sha256"]

cited = (repo / record["result"]).read_bytes()
assert (task / "rung0_delta_reproducibility.csv").read_bytes() == cited
print("the working copy in this folder is byte-identical to the cited copy")

## 2. What one correlation looks like

Each point is one gene: its fold change in the first group of plates against the second. Four
real conditions at the 5th, 25th, 50th and 95th percentile of the distribution above, then two
deliberately mismatched comparisons — the median condition's first group against a different
cell line's second group, once for the same drug, once for a different one.

Panel titles are computed from the points plotted. A typical condition (r = 0.109) is a slightly
tilted cloud; only the 95th percentile (0.354) is clearly elongated; mismatches are round. This
is what a ceiling of 0.238 looks like.

In [ ]:
profiles = pd.read_csv(task / "rung0_example_pair_profiles.csv.gz")
examples = pd.read_csv(task / "rung0_example_pair_index.csv")
print(examples.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(12, 7.4))
for ax, (_, ex) in zip(axes.ravel(), examples.iterrows(), strict=True):
    pts = profiles[profiles["example_id"] == ex["example_id"]]
    x, y = pts["lfc0"].to_numpy(float), pts["lfc1"].to_numpy(float)
    # the number in the title is recomputed here, from the points being drawn
    shown_r = float(np.corrcoef(x, y)[0, 1])
    assert abs(shown_r - ex["r_shown"]) < 5e-3, ex["example_id"]
    assert ex["r_shown"] == ex["r_full"], "every shared gene exported: the two must agree"
    lim = float(np.percentile(np.abs(np.concatenate([x, y])), 99.5))
    ax.scatter(x, y, s=2, alpha=0.18, color="steelblue", edgecolors="none")
    ax.plot([-lim, lim], [-lim, lim], color="0.6", lw=0.8, ls=":")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    if ex["kind"] == "matched":
        title = f"{ex['drug0']} on {ex['patient0']}\nsame condition, both halves: r = {shown_r:.3f}"
    else:
        title = (
            f"{ex['drug0']} on {ex['patient0']} vs {ex['drug1']} on {ex['patient1']}\n"
            f"mismatched ({ex['kind'].replace('_', ' ')}): r = {shown_r:.3f}"
        )
    ax.set_title(title, fontsize=8.5)
    ax.set_xlabel("half 0: log2 fold change", fontsize=8)
    ax.set_ylabel("half 1: log2 fold change", fontsize=8)
    ax.tick_params(labelsize=7)
fig.suptitle("The scatter behind the correlation: each point is one gene", fontsize=11)
fig.tight_layout()
plt.show()

Each chance floor is the mean of 500 mismatched-pair correlations like the last two
panels. The same-drug floor (0.079) sits above the different-drug floor (0.035): two cell lines
given the same drug still share that drug's generic response. The distributions overlap the
matched conditions; the separation is in the means, not in the tails.

In [ ]:
draws = pd.read_csv(task / "rung0_null_draws.csv")
for stratum, reported in (
    ("diff_drug", "null_diff_drug_mean_r"),
    ("same_drug", "null_same_drug_mean_r"),
    ("any_pair", "null_any_pair_mean_r"),
):
    d = draws[draws["stratum"] == stratum]["r"]
    print(
        f"{stratum:10s} {len(d)} draws  mean {d.mean():.4f}  sd {d.std():.4f}  "
        f"reported floor {headline[reported]}"
    )
    assert round(float(d.mean()), 3) == headline[reported]

fig, ax = plt.subplots(figsize=(9, 4.4))
bins = np.linspace(-0.25, 0.55, 80)
ax.hist(
    r, bins=bins, color="0.72", edgecolor="0.5", label=f"matched conditions (mean {np.mean(r):.3f})"
)
ax.hist(
    draws[draws["stratum"] == "same_drug"]["r"],
    bins=bins,
    alpha=0.65,
    color="darkorange",
    label="same drug, different line (500 draws)",
)
ax.hist(
    draws[draws["stratum"] == "diff_drug"]["r"],
    bins=bins,
    alpha=0.65,
    color="steelblue",
    label="different drug and line (500 draws)",
)
ax.axvline(np.mean(r), color="crimson", lw=2)
ax.set_xlabel("split-half correlation")
ax.set_ylabel("count")
ax.set_title("Matched conditions against the two chance floors, as distributions")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

## 3. What was measured, and on what

**Genes.** 13,886 of a declared 14,121-gene panel were present. The panel is the differential-
expression table's genes intersected with the gene sets of the datasets later rungs score. A
ceiling over all genes would price a target no rung reads: on an unpinned top-2,000
variable-gene panel the same data gives 0.299 rather than 0.109 — a different quantity, not a
better measurement. The panel file is pinned by checksum in the run record but not committed, so
14,121 is a recorded input property here, not a recomputable one.

**Drugs.** 32 compounds: those the later rungs' viability screens can score, intersected with
this screen by PubChem compound identifier. Same reasoning — the ceiling must be measured on the
drugs the ladder will actually score. The 32 identifiers resolve to 33 drug names, because
Trametinib also appears as a solvate variant, giving 33 × 50 cell lines = 1,650 candidate
conditions.

**Plates.** Ribociclib was run on a single plate throughout, so its 50 conditions cannot be
split at all: 1,650 − 50 = 1,600 scored. Plate depth also limits how a condition can be halved.
Three quarters of conditions have exactly three plates, which split one against two — unequal
groups, and only three distinct splits exist. The run uses one of them, chosen by a hash of the
plate identifier. Spearman-Brown (section 4) assumes two equal halves, so with a 1-vs-2 split it
is an approximation.

In [ ]:
pool = pd.read_csv(task / "rung0_pool_description.csv", keep_default_na=False)
scored = pool[(pool["n_plates_half0"] > 0) & (pool["n_plates_half1"] > 0)]
unsplittable = pool[(pool["n_plates_half0"] == 0) | (pool["n_plates_half1"] == 0)]

assert len(pool) == 1650 and pool["patient"].nunique() == 50 and pool["drug"].nunique() == 33
assert (pool["n_dose_levels"] == 3).all()
assert len(unsplittable) == 50 and set(unsplittable["drug"]) == {"Ribociclib"}
assert len(pool) - len(unsplittable) == int(headline["n_pairs"]) == 1600
print(
    f"{len(pool)} candidate - {len(unsplittable)} unsplittable = {len(scored)} scored "
    f"(reported n_pairs {int(headline['n_pairs'])})"
)

by_plates = scored.groupby("n_plates").size()
splits = {n: 2 ** (int(n) - 1) - 1 for n in by_plates.index}
for n, count in by_plates.items():
    print(
        f"  {n} plates: {count:5d} conditions ({100 * count / len(scored):4.1f}%), "
        f"{splits[n]:3d} distinct splits, 1 used"
    )

fig, ax = plt.subplots(figsize=(7.5, 3.6))
bars = ax.bar([str(n) for n in by_plates.index], by_plates.to_numpy(), color="0.7", edgecolor="0.3")
for bar, n, count in zip(bars, by_plates.index, by_plates.to_numpy(), strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        count + 15,
        f"{count}\n{splits[n]} splits",
        ha="center",
        fontsize=8,
    )
ax.set_xlabel("replicate plates per condition")
ax.set_ylabel("conditions")
ax.set_ylim(0, by_plates.max() * 1.25)
ax.set_title("Plate depth limits how many ways a condition can be halved")
fig.tight_layout()
plt.show()

Plate depth limits the *splits*, not the shuffles. The 500 mismatched-pair draws behind
each floor, and the 500 shuffles in section 5, are sampled from 1,600 × 1,599 = 2,558,400
ordered mismatched pairs. That sample size is a choice, and it sets the smallest reportable
p-value at 1/(1+500) = 0.002.

The screen itself is complete: every one of the 33 drugs was run on every one of the 50 cell
lines, at three doses each.

In [ ]:
print(
    len(pool),
    "conditions:",
    pool["patient"].nunique(),
    "cell lines x",
    pool["drug"].nunique(),
    "drug names (32 declared drugs + one solvate name variant)",
)
print("drugs delivered:", ", ".join(sorted(pool["drug"].unique())))
assert len(pool) == 1650 and pool["patient"].nunique() == 50 and pool["drug"].nunique() == 33
assert (pool["n_dose_levels"] == 3).all(), "dose design must be uniform: 3 levels everywhere"
print("every condition carries exactly 3 dose levels")

plates = pool.pivot(index="drug", columns="patient", values="n_plates")
assert plates.notna().all().all(), "the drug x line grid must be complete -- no missing conditions"
assert plates.shape == (33, 50)

fig, ax = plt.subplots(figsize=(11, 6.8))
im = ax.pcolormesh(plates.to_numpy(), cmap=plt.get_cmap("viridis", 7), vmin=0.5, vmax=7.5)
ax.set_yticks(np.arange(len(plates.index)) + 0.5)
ax.set_yticklabels(plates.index, fontsize=7)
ax.set_xticks([])
ax.set_xlabel(f"{plates.shape[1]} cell lines")
ax.set_title("The delivered screen: complete 33 x 50 grid, colored by replicate plates")
cbar = fig.colorbar(im, ticks=range(1, 8))
cbar.set_label("replicate plates")
fig.tight_layout()
plt.show()

The raw data cannot have changed either: the repository commits an inventory of all 1,026
downloaded files with their checksums, and the hash of that inventory is the data version the
run record pins.

In [ ]:
manifest = repo / "data" / "tranches" / "tahoe100m-pseudobulk-de.v1.manifest.txt"
registration_path = repo / "data" / "tranches" / "tahoe100m-pseudobulk-de.v1.json"
registration = json.loads(registration_path.read_text())

manifest_sha = hashlib.sha256(manifest.read_bytes()).hexdigest()
print("files in the inventory              :", len(manifest.read_text().splitlines()))
print("checksum of the inventory           :", manifest_sha)
print("data version at registration        :", registration["content_hash"])
print("data version the run record pinned  :", record["environment"]["data_commit"])
assert len(manifest.read_text().splitlines()) == 1026
assert manifest_sha == registration["content_hash"] == record["environment"]["data_commit"]

## 4. From half-data to the ceiling

Splitting the plates halves the data behind each measurement, so the correlation understates
what the full data supports. Spearman-Brown converts it: full = 2r/(1+r). The reported ceiling
must be that curve's value at the measured point.

In [ ]:
m = headline["splithalf_mean_r"]
print("measured half-data agreement :", m)
print("reported full-data ceiling   :", headline["spearman_brown_full"])

r = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(r, 2 * r / (1 + r), color="0.3", label="Spearman-Brown: full = 2r/(1+r)")
ax.plot([0, 1], [0, 1], ls=":", color="0.7", label="no correction")
ax.plot(
    m,
    headline["spearman_brown_full"],
    "o",
    color="crimson",
    ms=8,
    zorder=5,
    label=f"this measurement ({m} -> {headline['spearman_brown_full']})",
)
ax.plot(
    [m, m, 0],
    [0, headline["spearman_brown_full"], headline["spearman_brown_full"]],
    color="crimson",
    lw=0.8,
    ls="--",
)
ax.set_xlabel("split-half agreement (measured)")
ax.set_ylabel("full-data reliability (estimated)")
ax.set_title("The ceiling is the curve at the measured point")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

Conditions with larger responses should reproduce better; an assay where they did not
would be broken. Splitting conditions into thirds by response size is the built-in positive
control.

In [ ]:
# from section 1's raw values, not re-read from the summary row
terciles = [recomputed[f"splithalf_mean_r_tercile{t}"] for t in (1, 2, 3)]
print("tercile reliabilities, weakest to strongest responses:", terciles)
assert terciles[0] < terciles[1] < terciles[2]
assert terciles[0] > headline["null_same_drug_mean_r"] > headline["null_diff_drug_mean_r"]

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(
    ["weakest third\n(by effect size)", "middle third", "strongest third"],
    terciles,
    color="0.7",
    edgecolor="0.3",
)
ax.axhline(m, color="crimson", lw=1.8, label=f"overall mean {m}")
ax.axhline(
    headline["null_same_drug_mean_r"],
    ls="--",
    color="darkorange",
    label=f"same-drug chance floor {headline['null_same_drug_mean_r']}",
)
ax.axhline(
    headline["null_diff_drug_mean_r"],
    ls="--",
    color="steelblue",
    label=f"mismatched chance floor {headline['null_diff_drug_mean_r']}",
)
ax.set_ylabel("split-half reliability (mean Pearson r)")
ax.set_title("More signal, more reproducibility -- and every tercile clears both floors")
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout()
plt.show()

## 5. Does the pairing matter?

The reported p-values treat the mismatched draws as independent, though they reuse the same
half-profiles. Rather than argue the point, the pairing was shuffled: 500 times, every condition
was given a different condition's second group, and the mean correlation recomputed. Once
pooled, once within each comparison type the reported p-values use.

Grey is what the mean agreement looks like with the pairing destroyed; red is the true pairing.

In [ ]:
der = pd.read_csv(task / "rung0_derangement_summary.csv").iloc[0]
strata = [
    ("any-pair shuffle", "rung0_derangement_perm_means.csv", "observed_mean", ""),
    (
        "same-drug shuffle",
        "rung0_derangement_perm_means_same_drug.csv",
        "observed_mean_same_drug_rows",
        "_same_drug",
    ),
    (
        "diff-drug shuffle",
        "rung0_derangement_perm_means_diff_drug.csv",
        "observed_mean_diff_drug_rows",
        "_diff_drug",
    ),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharex=True)
for ax, (name, filename, observed_col, suffix) in zip(axes, strata, strict=True):
    perms = pd.read_csv(task / filename)["perm_mean"]
    observed = der[observed_col]
    p = (1 + int((perms >= observed).sum())) / (1 + len(perms))
    print(
        f"{name:18s}: {len(perms)} shuffles  mean {perms.mean():.4f}  "
        f"sd {perms.std():.4f}  max {perms.max():.4f} < observed {observed}  "
        f"exact p {p:.3f}"
    )
    assert len(perms) == 500 and observed > perms.max()
    assert round(perms.mean(), 4) == der["perm_mean_mean" + suffix]
    assert round(perms.std(), 4) == der["perm_mean_sd" + suffix]
    assert round(p, 3) == der["p_exact" + suffix]
    ax.hist(perms, bins=30, color="0.7", edgecolor="0.5")
    ax.axvline(observed, color="crimson", lw=2)
    ax.set_title(f"{name}\n500 shuffled nulls vs observed (p = {p:.3f})", fontsize=9)
    ax.set_xlabel("mean mismatched correlation")
axes[0].set_ylabel("shuffles")
fig.tight_layout()
plt.show()

The design effect is the shuffled variance divided by the variance the independence
shortcut assumed. Below one means the shortcut was conservative. The per-stratum values need
each stratum's pooled standard error, which only the cluster run holds; they are checked in the
scripted battery.

The shuffled floors also land on the floors the published row reports, by a different sampling
mechanism.

In [ ]:
any_perms = pd.read_csv(task / "rung0_derangement_perm_means.csv")["perm_mean"]
design_effect = any_perms.var() / der["se_iid_pool"] ** 2
print("design effect =", round(design_effect, 3), "  reported:", der["design_effect"])
assert abs(design_effect - der["design_effect"]) < 0.02 and design_effect < 1

print(
    "same-drug floor: derangement",
    der["perm_mean_mean_same_drug"],
    " bootstrap",
    headline["null_same_drug_mean_r"],
)
print(
    "diff-drug floor: derangement",
    der["perm_mean_mean_diff_drug"],
    " bootstrap",
    headline["null_diff_drug_mean_r"],
)
assert abs(der["perm_mean_mean_same_drug"] - headline["null_same_drug_mean_r"]) < 0.0015
assert abs(der["perm_mean_mean_diff_drug"] - headline["null_diff_drug_mean_r"]) < 0.0015

## 6. Where the reproducibility sits

Each panel gene correlated across conditions between the two plate groups. Nearly the whole
panel is positive, and the top of the ranking is heat-shock and immediate-early stress genes — a
generic perturbation response that reproduces regardless of drug or cell line.

In [ ]:
pg = pd.read_csv(task / "rung0_per_gene_reliability.csv")
finite = pg[np.isfinite(pg["r"])]
top5 = pg.nlargest(5, "r")

print(len(pg), "genes,", len(finite), "with a finite value,", len(pg) - len(finite), "without")
print(
    f"positive fraction {100 * (finite['r'] > 0).mean():.1f}%  "
    f"median {finite['r'].median():.3f}  "
    f"quartiles {finite['r'].quantile(0.25):.3f}-{finite['r'].quantile(0.75):.3f}"
)
print(top5.to_string(index=False))
assert len(pg) == 13886 and len(finite) == 13759
assert abs((finite["r"] > 0).mean() - 0.970) <= 0.0005
assert abs(finite["r"].median() - 0.146) <= 0.0005
assert abs(finite["r"].quantile(0.25) - 0.089) <= 0.0005
assert abs(finite["r"].quantile(0.75) - 0.230) <= 0.0005
assert list(top5["gene"]) == ["HSP90AA1", "EGR1", "HSPA1B", "HSPH1", "PLEC"]

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.hist(finite["r"], bins=90, color="0.7", edgecolor="0.6")
ax.axvline(0, color="k", lw=1)
ax.axvline(
    finite["r"].median(), color="crimson", lw=1.8, label=f"median {finite['r'].median():.3f}"
)
for (_, g), y in zip(top5.iterrows(), (900, 740, 580, 420, 260), strict=True):
    ax.plot([g["r"], g["r"]], [0, y - 40], color="0.45", lw=0.6)
    ax.annotate(g["gene"], (g["r"], y), fontsize=8, ha="center")
ax.set_xlabel("per-gene split-half reliability r (across 1,600 conditions)")
ax.set_ylabel("genes")
ax.set_title("13,759 panel genes: 97.0% right of zero; stress-response genes lead")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 7. The summary matches the artifacts

Each number in `summary.md`'s evidence table, parsed from the document and compared with the
artifact it came from. The prose paragraphs' numbers are checked the same way in the scripted
battery.

In [ ]:
text = (task / "summary.md").read_text()


def numbers(label):
    row = next(ln for ln in text.splitlines() if ln.startswith("|") and label in ln)
    cell = row.rsplit("|", 2)[-2]
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", cell)]


assert numbers("Conditions scored") == [headline["n_pairs"]]
# the declared 14,121 is hash-pinned, not recomputable locally (see the intro)
assert numbers("Panel genes present")[0] == headline["n_genes"]
assert numbers("Split-half reliability") == [
    headline["splithalf_mean_r"],
    headline["splithalf_median_r"],
    headline["splithalf_q1_r"],
    headline["splithalf_q3_r"],
]
assert numbers("Spearman-Brown") == [headline["spearman_brown_full"]]
assert numbers("positive reliability") == [round(100 * headline["frac_pos"], 1)]
assert numbers("Mismatched-condition floor")[-1] == headline["null_diff_drug_mean_r"]
assert numbers("Same-drug floor")[-1] == headline["null_same_drug_mean_r"]
assert numbers("Significance")[0] == headline["p_vs_null"] == headline["p_vs_same_drug"]
assert numbers("Smallest detectable")[-2:] == [
    round(headline["mde_80_vs_diff_drug"], 3),
    round(headline["mde_80_vs_same_drug"], 3),
]
print("every evidence-table number matches its artifact")

## 8. The code that produced the numbers

The known-answer controls: synthetic data with a reliability of 0.8 planted must come back as
0.8 through the real measurement, signal-free data must come back null, the power calculation
must match the closed-form answer, and the one statistical defect this project has shipped
(comparing an aggregate against single draws) is pinned by a test showing the wrong form
failing.

In [ ]:
result = subprocess.run(
    ["uv", "run", "pytest", "-m", "known_answer", "-q"], cwd=repo, capture_output=True, text=True
)
print(result.stdout[-2500:] + result.stderr[-500:])
assert result.returncode == 0, "known-answer controls failed"

Finally the scripted battery — the same checks plus those needing the cluster-only
values — which must agree with everything above.

In [ ]:
battery = subprocess.run(
    ["uv", "run", "python", "scripts/verify_rung0.py"], cwd=repo, capture_output=True, text=True
)
print(battery.stdout[-600:])
assert battery.returncode == 0, "the scripted battery disagrees with the cells above"
print("battery agrees with the cells above")